# Re-annoter le 800k (vraies parties externes) et le 110k (notre dataset) avec le professeur bidoua-aware

Contexte (juillet 2026) : le professeur (`DeepTeacher`) valorise desormais
les accumulations sures (5+ graines, "bidoua"/"Yinda") dans son
evaluation de repli -- correctif deja actif dans le code (rien a changer
ici), mais les labels DEJA ecrits sur disque restent geles a l'ancienne
version tant qu'on ne relance pas l'annotation. Ce notebook re-annote
DEUX datasets separement :

- **Section A** : le dataset externe (883k positions de vraies parties,
  ex-labels minimax "insane") -- `reannotate_external_dataset.py`.
- **Section B** : notre propre dataset `dataset_v003_110k` (10k+100k
  generes par notre prof) -- `reannotate_own_dataset.py`.

**ATTENTION CACHE (piege reel)** : le cache d'annotations est indexe par
position + config du professeur, PAS par version du code d'evaluation --
reutiliser un `CACHE_DIR` deja peuple AVANT ce correctif renverrait
silencieusement les anciennes annotations. Les noms de dossiers
ci-dessous portent tous un suffixe `_bidoua_v1` : ne jamais les faire
pointer vers un ancien dossier de cache.

**Le cache SQLite doit vivre sur Drive** (pas sur le disque Colab,
ephemere) : si la session se coupe, relancer ce notebook reprend
exactement ou ca s'est arrete, sans rien recalculer.

**Debit mesure jusqu'ici (a confirmer sur CE Colab, pas a supposer)** :
~15-16s/position sur un premier echantillon du dataset externe (mesure
locale, ancienne campagne), ~61s/position sur un pilote de 48 positions
du dataset 110k (mesure locale, juillet 2026, Mac 8 coeurs -- le budget
de 60s est presque systematiquement epuise sur ce dataset). Ces deux
chiffres ne sont PAS interchangeables (positions differentes) : toujours
piloter sur CE dataset et CET environnement avant de lancer le volume
complet -- meme discipline que pour les paliers GCP precedents.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/SongoFish'  # ajuster si besoin
import os
assert os.path.isdir(DRIVE_ROOT), f"Dossier introuvable : {DRIVE_ROOT}"

# Chemins sous data/ (PAS datasets/ -- corrige juillet 2026 : c'est le
# chemin reellement utilise par la synchro rclone de ce projet, verifie
# via `rclone lsd songo_drive:SongoFish/data`).
EXTERNAL_INPUT_DIR = f'{DRIVE_ROOT}/data/dataset_full_matrix_merged_all_colabs'
EXTERNAL_OUT_DIR = f'{DRIVE_ROOT}/data/dataset_external_reannotated_bidoua_v1'
EXTERNAL_CACHE_DIR = f'{EXTERNAL_OUT_DIR}/annotation_cache'  # SUR DRIVE : survit aux deconnexions

OWN_INPUT_DIR = f'{DRIVE_ROOT}/data/dataset_v003_110k'
OWN_OUT_DIR = f'{DRIVE_ROOT}/data/dataset_v003_110k_bidoua_v1'
OWN_CACHE_DIR = f'{OWN_OUT_DIR}/annotation_cache'  # SUR DRIVE : survit aux deconnexions

for path, label in [(EXTERNAL_INPUT_DIR, 'dataset externe'), (OWN_INPUT_DIR, 'dataset 110k')]:
    assert os.path.isdir(path), f"{label} introuvable a {path} -- verifier la synchro Drive avant de continuer"


In [ ]:
!git clone https://github.com/GlennEriss/songo-fish.git /content/songo-fish
%cd /content/songo-fish
!pip install -q -e ".[train]"


In [ ]:
import multiprocessing
print('coeurs disponibles sur cette session Colab:', multiprocessing.cpu_count())


## Section A -- dataset externe (883k, vraies parties)

### A.1 -- pilote (mesurer le debit reel avant de s'engager)

In [ ]:
!.venv/bin/python apps/trainer/scripts/reannotate_external_dataset.py \
    --input-dir "{EXTERNAL_INPUT_DIR}" \
    --out-dir /content/pilot_external_bidoua \
    --cache-dir "{EXTERNAL_CACHE_DIR}" \
    --limit 100 --num-workers 2


### A.2 -- volume complet (une fois le debit du pilote juge acceptable)

Retirer `--limit` pour prendre tout. Le cache etant sur Drive, relancer
cette meme cellule apres une deconnexion reprend sans tout refaire.

In [ ]:
!.venv/bin/python apps/trainer/scripts/reannotate_external_dataset.py \
    --input-dir "{EXTERNAL_INPUT_DIR}" \
    --out-dir "{EXTERNAL_OUT_DIR}" \
    --cache-dir "{EXTERNAL_CACHE_DIR}" \
    --num-workers 2


## Section B -- notre dataset (110k, deja au format interne)

### B.1 -- pilote (mesurer le debit reel avant de s'engager)

In [ ]:
!.venv/bin/python apps/trainer/scripts/reannotate_own_dataset.py \
    --input-dir "{OWN_INPUT_DIR}" \
    --out-dir /content/pilot_110k_bidoua \
    --cache-dir "{OWN_CACHE_DIR}" \
    --limit 30 --num-workers 2


### B.2 -- volume complet (une fois le debit du pilote juge acceptable)

Retirer `--limit` pour prendre tout. Le cache etant sur Drive, relancer
cette meme cellule apres une deconnexion reprend sans tout refaire.

In [ ]:
!.venv/bin/python apps/trainer/scripts/reannotate_own_dataset.py \
    --input-dir "{OWN_INPUT_DIR}" \
    --out-dir "{OWN_OUT_DIR}" \
    --cache-dir "{OWN_CACHE_DIR}" \
    --num-workers 2


## Etape suivante (pas encore decidee) -- fusion

Une fois les DEUX re-annotations terminees (`EXTERNAL_OUT_DIR` et
`OWN_OUT_DIR`), reste a decider comment les combiner avec le reste
(`dataset_v004_290k`, `dataset_v005_400k`, pas encore re-annotes) --
volontairement pas fait automatiquement ici, c'est une decision a prendre
une fois les deux volumes reellement re-annotes et leurs metriques
regardees (pas un merge aveugle). Exemple de commande (a ajuster) :

```python
from pathlib import Path
from songo_ai.dataset import merge_releases

manifest = merge_releases(
    [Path(OWN_OUT_DIR), Path(EXTERNAL_OUT_DIR)],
    Path(f'{DRIVE_ROOT}/data/dataset_v007_bidoua_merged'),
    deduplicate=True,
)
print(manifest['total_positions'], manifest['counts_per_split'])
```